In [ ]:
import sys
sys.path.append('benchmark_dim_red')
import os
os.environ['TMP'] = '/tmp'
import plotly.express as px
import lzma
import pickle
from methods import methods, with_weight
import numpy as np
import pandas as pd
from ioh import get_problem, ProblemType, ProblemClass
from sklearn.model_selection import cross_val_score, KFold
from sklearn.metrics import r2_score, mean_squared_error
from pflacco.sampling import create_initial_sample
from sklearn.preprocessing import MinMaxScaler
from real_world_benchs.real_world_benchs import *
from functools import partial
from scipy.stats import  variation

benchs = {
    "Rover": Rover, 
    "SVM": SVM, 
    "Mujoco_ant": partial(Mujoco, 'ant'), 
    "Mujoco_half_cheetah": partial(Mujoco, 'half_cheetah'), 
    "Mujoco_hopper": partial(Mujoco, 'hopper'), 
    "Mujoco_humanoid": partial(Mujoco, 'humanoid'), 
    "Mujoco_swimmer": partial(Mujoco, 'swimmer'),
    "Mujoco_walker_2d": partial(Mujoco, 'walker_2d'), 
    "Mopta": Mopta, 
    "LassoReal_leukemia": partial(LassoReal, 'leukemia'), 
    "LassoReal_dna": partial(LassoReal, 'dna'),
    "LassoSynthetic_synt_simple": partial(LassoSynthetic, 'synt_simple'),
    "LassoSynthetic_synt_medium": partial(LassoSynthetic, 'synt_medium'), 
    "LassoSynthetic_synt_high": partial(LassoSynthetic, 'synt_high'),
    "LassoSynthetic_synt_hard": partial(LassoSynthetic, 'synt_hard')
}

In [ ]:
for sample in [100, 200, 500, 1000]:
    for fid in range(1, 25):
        for inpt_dim in [20, 40, 60, 120, 240, 500, 1000]:
            for red_dim in [5, 10, 20, 40]:
                if inpt_dim <= red_dim:
                    continue
                df = pd.read_parquet(f'benchmark_dim_red/agg_res/epm/{sample}_{fid}_{inpt_dim}.parquet.xz')

                df = df[df['red_dim'] == red_dim]
                
                df['mse'] = np.sqrt(df['mse'])
                df = df.groupby(['method_name', 'cv']).mean().reset_index()

                fig = px.box(df, x='method_name', y='mse', color='method_name',
                             color_discrete_map={
                                    'PCA_linear':px.colors.qualitative.Plotly[0],
                                    'PCA_rbf':px.colors.qualitative.Dark24[5],
                                    'PCA_sigmoid':px.colors.qualitative.Safe[10],
                                    'PCA_cosine':px.colors.qualitative.Safe[9],
                                    'PCA_poly': px.colors.qualitative.Safe[3],
                                    'IPCA':px.colors.qualitative.Safe[5],
                                    'FastICA':px.colors.qualitative.Plotly[1],
                                    'TruncatedSVD':px.colors.qualitative.Plotly[2],
                                    'lolP':px.colors.qualitative.Plotly[3],
                                    'isomap':px.colors.qualitative.Plotly[4],
                                    'lle':px.colors.qualitative.Plotly[5],
                                    'pls_reg':px.colors.qualitative.Plotly[6],
                                    'DictionaryLearning':px.colors.qualitative.Plotly[7],
                                    'sparsePCA':px.colors.qualitative.Plotly[8],
                                    'FA':px.colors.qualitative.Plotly[9],
                                    'BernoulliRBM':px.colors.qualitative.G10[4],
                                    'GaussianRandomProjection':px.colors.qualitative.G10[8],
                                    'SparseRandomProjection':px.colors.qualitative.G10[9],
                                })
                fig.update_xaxes(categoryorder='array', 
                                 categoryarray=['PCA_linear', 'PCA_rbf', 'PCA_sigmoid', 
                                                'PCA_cosine', 'PCA_poly', 'IPCA', 'FastICA', 
                                                'DictionaryLearning', 'sparsePCA', 'GaussianRandomProjection', 'SparseRandomProjection', 'lolP', 'TruncatedSVD',
                                                'isomap', 'lle', 'pls_reg', 'FA', 'BernoulliRBM'], title=None)
                fig.update_layout(
                    xaxis = dict(
                        tickmode = 'array',
                        tickvals = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24],
                        ticktext =['PCA_linear', 'PCA_rbf', 'PCA_sigmoid', 
                                                'PCA_cosine', 'PCA_poly', 'IPCA', 'FastICA', 
                                                'DictionaryLearning', 'sparsePCA', 'GaussianRandomProjection', 'SparseRandomProjection', 'lolP', 'TruncatedSVD',
                                                'isomap', 'lle', 'pls_reg', 'FA', 'BernoulliRBM']
                    )
                )

                fig.update_layout(title_text=None,
                        template='plotly_white',
                        showlegend=False,
                        legend_title=None,
                        width=200, height=120, margin=dict(l=0, r=0, t=0, b=0),
                        font=dict(
                            family="Times New Roman",
                            size=10,
                            color="Black"
                        ),
                        title_font=dict(
                            family="Times New Roman",
                            size=10,
                            color="Black"
                        ),
                        yaxis_title="RMSE",
                        )
                fig.update_xaxes(showticklabels=False)
                fig.write_image(f'benchmark_dim_red/figures/epm/{sample}_{fid}_{inpt_dim}_{red_dim}.pdf')


In [ ]:
renames = {
    'BernoulliRBM': 'BernoulliRBM',
    'DictionaryLearning': 'DictionaryLearning',
    'FA': 'FA',
    'FastICA': 'FastICA',
    'GaussianRandomProjection': 'Gaussian Random Projection',
    'IPCA': 'IPCA',
    'PCA_cosine': 'PCA (Cosine)',
    'PCA_linear': 'PCA (Linear)',
    'PCA_poly': 'PCA (Polynomial)',
    'PCA_rbf': 'PCA (RBF)',
    'PCA_sigmoid': 'PCA (Sigmoid)',
    'SparseRandomProjection': 'Sparse Random Projection',
    'TruncatedSVD': 'Truncated SVD',
    'isomap': 'Isomap',
    'lle': 'LLE',
    'lolP': 'LoLP',
    'pls_reg': 'PLS Regression',
    'sparsePCA': 'Sparse PCA'
}
df.replace({'method_name': renames}, inplace=True)
fig = px.box(df, x='method_name', y='mse', color='method_name',
                             color_discrete_map={
                                    'PCA (Linear)':px.colors.qualitative.Plotly[0],
                                    'PCA (RBF)':px.colors.qualitative.Dark24[5],
                                    'PCA (Sigmoid)':px.colors.qualitative.Safe[10],
                                    'PCA (Cosine)':px.colors.qualitative.Safe[9],
                                    'PCA (Polynomial)': px.colors.qualitative.Safe[3],
                                    'IPCA':px.colors.qualitative.Safe[5],
                                    'FastICA':px.colors.qualitative.Plotly[1],
                                    'Truncated SVD':px.colors.qualitative.Plotly[2],
                                    'LoLP':px.colors.qualitative.Plotly[3],
                                    'Isomap':px.colors.qualitative.Plotly[4],
                                    'LLE':px.colors.qualitative.Plotly[5],
                                    'PLS Regression':px.colors.qualitative.Plotly[6],
                                    'DictionaryLearning':px.colors.qualitative.Plotly[7],
                                    'Sparse PCA':px.colors.qualitative.Plotly[8],
                                    'FA':px.colors.qualitative.Plotly[9],
                                    'BernoulliRBM':px.colors.qualitative.G10[4],
                                    'Gaussian Random Projection':px.colors.qualitative.G10[8],
                                    'Sparse Random Projection':px.colors.qualitative.G10[9],
                                })
fig.add_vline(x=10.5, line_width=1, line_dash="dash", line_color="grey")

fig.update_layout(title_text=None,
        template='plotly_white',
        showlegend=False,
        legend_title=None,
        width=100, height=300, margin=dict(l=0, r=0, t=0, b=0),
        font=dict(
            family="Times New Roman",
            size=10,
            color="Black"
        ),
        title_font=dict(
            family="Times New Roman",
            size=10,
            color="Black"
        ),
        yaxis_title="RMSE",
        )

fig.update_layout(legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="center", x=0.4), width=400, height=400, showlegend=True, legend_font_size=10)
fig.update_layout(legend_tracegroupgap=1, legend_indentation=0)
fig.write_image("benchmark_dim_red/paper/figures/legend_small.pdf")
fig.show()

In [ ]:
df["method_name"].unique()

In [ ]:
for sample in [100, 200, 500, 1000]:
    for inpt_dim in [20, 40, 60, 120, 240, 500, 1000]:
        for red_dim in [5, 10, 20, 40]:
            if inpt_dim <= red_dim:
                continue
            dfs = []
            for fid in range(1, 25):

                df = pd.read_parquet(f'benchmark_dim_red/agg_res/epm/{sample}_{fid}_{inpt_dim}.parquet.xz')
                
                df = df[df['red_dim'] == red_dim]
                df['mse'] = np.sqrt(df['mse'])
                df = df.groupby(['method_name', 'cv']).mean().reset_index()
                df['rank'] =1 - ( df['mse'].max() - df['mse'] ) /( df['mse'].max() - df['mse'].min())
                dfs.append(df)
            df = pd.concat(dfs)
            fig = px.box(df, x='method_name', y='rank', color='method_name',
                            color_discrete_map={
                                'PCA_linear':px.colors.qualitative.Plotly[0],
                                'PCA_rbf':px.colors.qualitative.Dark24[5],
                                'PCA_sigmoid':px.colors.qualitative.Safe[10],
                                'PCA_cosine':px.colors.qualitative.Safe[9],
                                'PCA_poly': px.colors.qualitative.Safe[3],
                                'IPCA':px.colors.qualitative.Safe[5],
                                'FastICA':px.colors.qualitative.Plotly[1],
                                'TruncatedSVD':px.colors.qualitative.Plotly[2],
                                'lolP':px.colors.qualitative.Plotly[3],
                                'isomap':px.colors.qualitative.Plotly[4],
                                'lle':px.colors.qualitative.Plotly[5],
                                'pls_reg':px.colors.qualitative.Plotly[6],
                                'DictionaryLearning':px.colors.qualitative.Plotly[7],
                                'sparsePCA':px.colors.qualitative.Plotly[8],
                                'FA':px.colors.qualitative.Plotly[9],
                                'BernoulliRBM':px.colors.qualitative.G10[4],
                                'GaussianRandomProjection':px.colors.qualitative.G10[8],
                                'SparseRandomProjection':px.colors.qualitative.G10[9],
                            }, log_y=True)
            fig.update_xaxes(categoryorder='array', 
                                categoryarray=['PCA_linear', 'PCA_rbf', 'PCA_sigmoid', 
                                            'PCA_cosine', 'PCA_poly', 'IPCA', 'FastICA', 
                                            'DictionaryLearning', 'sparsePCA', 'GaussianRandomProjection', 'SparseRandomProjection', 'lolP', 'TruncatedSVD',
                                            'isomap', 'lle', 'pls_reg', 'FA', 'BernoulliRBM'], title=None)
            fig.update_layout(
                xaxis = dict(
                    tickmode = 'array',
                    tickvals = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24],
                    ticktext =['PCA_linear', 'PCA_rbf', 'PCA_sigmoid', 
                                            'PCA_cosine', 'PCA_poly', 'IPCA', 'FastICA', 
                                            'DictionaryLearning', 'sparsePCA', 'GaussianRandomProjection', 'SparseRandomProjection', 'lolP', 'TruncatedSVD',
                                            'isomap', 'lle', 'pls_reg', 'FA', 'BernoulliRBM']
                )
            )

            fig.update_layout(title_text=None,
                    template='plotly_white',
                    showlegend=False,
                    legend_title=None,
                    width=200, height=200, margin=dict(l=0, r=0, t=0, b=0),
                    font=dict(
                        family="Times New Roman",
                        size=10,
                        color="Black"
                    ),
                    title_font=dict(
                        family="Times New Roman",
                        size=10,
                        color="Black"
                    ),
                    yaxis_title="RMSE",
                    )
            fig.update_xaxes(showticklabels=False)
            fig.write_image(f'benchmark_dim_red/figures/epm/agg_{sample}_{inpt_dim}_{red_dim}.pdf')


In [ ]:
for sample in [100, 200, 500, 1000]:
    for red_dim in [5, 10, 20, 40]:
        dfs = []
        for inpt_dim in [20, 40, 60, 120, 240, 500, 1000]:

            if inpt_dim <= red_dim:
                continue
            
            for fid in range(1, 25):

                df = pd.read_parquet(f'benchmark_dim_red/agg_res/epm/{sample}_{fid}_{inpt_dim}.parquet.xz')
                
                df = df[df['red_dim'] == red_dim]
                df['mse'] = np.sqrt(df['mse'])
                min_mse = df['mse'].min()
                max_mse = df['mse'].max()
                df = df.groupby(['method_name', 'cv']).mean().reset_index()
                df['rank'] =1 - ( df['mse'].max() - df['mse'] ) /( df['mse'].max() - df['mse'].min())
                df['rank_coefficient_of_variation'] = variation(df['mse'])
                dfs.append(df)
        df = pd.concat(dfs)
        df['rank_weight'] = df['rank_coefficient_of_variation'] / df['rank_coefficient_of_variation'].sum()
        fig = px.box(df, x='method_name', y='rank', color='method_name',
                        color_discrete_map={
                            'PCA_linear':px.colors.qualitative.Plotly[0],
                            'PCA_rbf':px.colors.qualitative.Dark24[5],
                            'PCA_sigmoid':px.colors.qualitative.Safe[10],
                            'PCA_cosine':px.colors.qualitative.Safe[9],
                            'PCA_poly': px.colors.qualitative.Safe[3],
                            'IPCA':px.colors.qualitative.Safe[5],
                            'FastICA':px.colors.qualitative.Plotly[1],
                            'TruncatedSVD':px.colors.qualitative.Plotly[2],
                            'lolP':px.colors.qualitative.Plotly[3],
                            'isomap':px.colors.qualitative.Plotly[4],
                            'lle':px.colors.qualitative.Plotly[5],
                            'pls_reg':px.colors.qualitative.Plotly[6],
                            'DictionaryLearning':px.colors.qualitative.Plotly[7],
                            'sparsePCA':px.colors.qualitative.Plotly[8],
                            'FA':px.colors.qualitative.Plotly[9],
                            'BernoulliRBM':px.colors.qualitative.G10[4],
                            'GaussianRandomProjection':px.colors.qualitative.G10[8],
                            'SparseRandomProjection':px.colors.qualitative.G10[9],
                        }, 
                        )
        fig.update_xaxes(categoryorder='array', 
                            categoryarray=['PCA_linear', 'PCA_rbf', 'PCA_sigmoid', 
                                        'PCA_cosine', 'PCA_poly', 'IPCA', 'FastICA', 
                                        'DictionaryLearning', 'sparsePCA', 'GaussianRandomProjection', 'SparseRandomProjection', 'lolP', 'TruncatedSVD',
                                        'isomap', 'lle', 'pls_reg', 'FA', 'BernoulliRBM'], title=None)
        fig.update_layout(
            xaxis = dict(
                tickmode = 'array',
                tickvals = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24],
                ticktext =['PCA_linear', 'PCA_rbf', 'PCA_sigmoid', 
                                        'PCA_cosine', 'PCA_poly', 'IPCA', 'FastICA', 
                                        'DictionaryLearning', 'sparsePCA', 'GaussianRandomProjection', 'SparseRandomProjection', 'lolP', 'TruncatedSVD',
                                        'isomap', 'lle', 'pls_reg', 'FA', 'BernoulliRBM']
            )
        )

        fig.update_layout(title_text=None,
                template='plotly_white',
                showlegend=False,
                legend_title=None,
                width=200, height=120, margin=dict(l=0, r=0, t=0, b=0),
                font=dict(
                    family="Times New Roman",
                    size=10,
                    color="Black"
                ),
                title_font=dict(
                    family="Times New Roman",
                    size=10,
                    color="Black"
                ),
                yaxis_title="Mean Rank",
                )
        fig.update_xaxes(showticklabels=False)
        fig.write_image(f'benchmark_dim_red/figures/epm/agg_sample_red_{sample}_{red_dim}.pdf')


In [ ]:
for sample in [100, 200, 500, 1000]:
    dfs = []
    for red_dim in [5, 10, 20, 40]:
        
        for inpt_dim in [20, 40, 60, 120, 240, 500, 1000]:

            if inpt_dim <= red_dim:
                continue
            
            for fid in range(1, 25):

                df = pd.read_parquet(f'benchmark_dim_red/agg_res/epm/{sample}_{fid}_{inpt_dim}.parquet.xz')
                
                df = df[df['red_dim'] == red_dim]
                df['mse'] = np.sqrt(df['mse'])
                df = df.groupby(['method_name', 'cv']).mean().reset_index()
                df['rank'] =1 - ( df['mse'].max() - df['mse'] ) /( df['mse'].max() - df['mse'].min())
                df['rank_coefficient_of_variation'] = variation(df['mse'])
                dfs.append(df)
    df = pd.concat(dfs)
    fig = px.box(df, x='method_name', y='mse', color='method_name',
                    color_discrete_map={
                        'PCA_linear':px.colors.qualitative.Plotly[0],
                        'PCA_rbf':px.colors.qualitative.Dark24[5],
                        'PCA_sigmoid':px.colors.qualitative.Safe[10],
                        'PCA_cosine':px.colors.qualitative.Safe[9],
                        'PCA_poly': px.colors.qualitative.Safe[3],
                        'IPCA':px.colors.qualitative.Safe[5],
                        'FastICA':px.colors.qualitative.Plotly[1],
                        'TruncatedSVD':px.colors.qualitative.Plotly[2],
                        'lolP':px.colors.qualitative.Plotly[3],
                        'isomap':px.colors.qualitative.Plotly[4],
                        'lle':px.colors.qualitative.Plotly[5],
                        'pls_reg':px.colors.qualitative.Plotly[6],
                        'DictionaryLearning':px.colors.qualitative.Plotly[7],
                        'sparsePCA':px.colors.qualitative.Plotly[8],
                        'FA':px.colors.qualitative.Plotly[9],
                        'BernoulliRBM':px.colors.qualitative.G10[4],
                        'GaussianRandomProjection':px.colors.qualitative.G10[8],
                        'SparseRandomProjection':px.colors.qualitative.G10[9],
                    }, log_y=True)
    fig.update_xaxes(categoryorder='array', 
                        categoryarray=['PCA_linear', 'PCA_rbf', 'PCA_sigmoid', 
                                    'PCA_cosine', 'PCA_poly', 'IPCA', 'FastICA', 
                                    'DictionaryLearning', 'sparsePCA', 'GaussianRandomProjection', 'SparseRandomProjection', 'lolP', 'TruncatedSVD',
                                    'isomap', 'lle', 'pls_reg', 'FA', 'BernoulliRBM'], title=None)
    fig.update_layout(
        xaxis = dict(
            tickmode = 'array',
            tickvals = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24],
            ticktext =['PCA_linear', 'PCA_rbf', 'PCA_sigmoid', 
                                    'PCA_cosine', 'PCA_poly', 'IPCA', 'FastICA', 
                                    'DictionaryLearning', 'sparsePCA', 'GaussianRandomProjection', 'SparseRandomProjection', 'lolP', 'TruncatedSVD',
                                    'isomap', 'lle', 'pls_reg', 'FA', 'BernoulliRBM']
        )
    )
    fig.add_vline(x=10.5, line_width=1, line_dash="dash", line_color="grey")

    fig.update_layout(title_text=None,
            template='plotly_white',
            showlegend=False,
            legend_title=None,
            width=200, height=200, margin=dict(l=0, r=0, t=0, b=0),
            font=dict(
                family="Times New Roman",
                size=10,
                color="Black"
            ),
            title_font=dict(
                family="Times New Roman",
                size=10,
                color="Black"
            ),
            yaxis_title="RMSE",
            )
    fig.update_xaxes(showticklabels=False)
    fig.write_image(f'benchmark_dim_red/figures/epm/agg_sample_{sample}.pdf')


In [ ]:

for red_dim in [5, 10, 20, 40]:
    dfs = []
    for sample in [100, 200, 500, 1000]:
        
        for inpt_dim in [20, 40, 60, 120, 240, 500, 1000]:

            if inpt_dim <= red_dim:
                continue
            
            for fid in range(1, 25):

                df = pd.read_parquet(f'benchmark_dim_red/agg_res/epm/{sample}_{fid}_{inpt_dim}.parquet.xz')
                
                df = df[df['red_dim'] == red_dim]
                df['mse'] = np.sqrt(df['mse'])
                df = df.groupby(['method_name', 'cv']).mean().reset_index()
                df['rank'] =1 - ( df['mse'].max() - df['mse'] ) /( df['mse'].max() - df['mse'].min())
                df['rank_coefficient_of_variation'] = variation(df['mse'])
                dfs.append(df)
    df = pd.concat(dfs)
    fig = px.box(df, x='method_name', y='mse', color='method_name',
                    color_discrete_map={
                        'PCA_linear':px.colors.qualitative.Plotly[0],
                        'PCA_rbf':px.colors.qualitative.Dark24[5],
                        'PCA_sigmoid':px.colors.qualitative.Safe[10],
                        'PCA_cosine':px.colors.qualitative.Safe[9],
                        'PCA_poly': px.colors.qualitative.Safe[3],
                        'IPCA':px.colors.qualitative.Safe[5],
                        'FastICA':px.colors.qualitative.Plotly[1],
                        'TruncatedSVD':px.colors.qualitative.Plotly[2],
                        'lolP':px.colors.qualitative.Plotly[3],
                        'isomap':px.colors.qualitative.Plotly[4],
                        'lle':px.colors.qualitative.Plotly[5],
                        'pls_reg':px.colors.qualitative.Plotly[6],
                        'DictionaryLearning':px.colors.qualitative.Plotly[7],
                        'sparsePCA':px.colors.qualitative.Plotly[8],
                        'FA':px.colors.qualitative.Plotly[9],
                        'BernoulliRBM':px.colors.qualitative.G10[4],
                        'GaussianRandomProjection':px.colors.qualitative.G10[8],
                        'SparseRandomProjection':px.colors.qualitative.G10[9],
                    }, log_y=True)
    fig.update_xaxes(categoryorder='array', 
                        categoryarray=['PCA_linear', 'PCA_rbf', 'PCA_sigmoid', 
                                    'PCA_cosine', 'PCA_poly', 'IPCA', 'FastICA', 
                                    'DictionaryLearning', 'sparsePCA', 'GaussianRandomProjection', 'SparseRandomProjection', 'lolP', 'TruncatedSVD',
                                    'isomap', 'lle', 'pls_reg', 'FA', 'BernoulliRBM'], title=None)
    fig.update_layout(
        xaxis = dict(
            tickmode = 'array',
            tickvals = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24],
            ticktext =['PCA_linear', 'PCA_rbf', 'PCA_sigmoid', 
                                    'PCA_cosine', 'PCA_poly', 'IPCA', 'FastICA', 
                                    'DictionaryLearning', 'sparsePCA', 'GaussianRandomProjection', 'SparseRandomProjection', 'lolP', 'TruncatedSVD',
                                    'isomap', 'lle', 'pls_reg', 'FA', 'BernoulliRBM']
        )
    )
    fig.add_vline(x=10.5, line_width=1, line_dash="dash", line_color="grey")

    fig.update_layout(title_text=None,
            template='plotly_white',
            showlegend=False,
            legend_title=None,
            width=200, height=200, margin=dict(l=0, r=0, t=0, b=0),
            font=dict(
                family="Times New Roman",
                size=10,
                color="Black"
            ),
            title_font=dict(
                family="Times New Roman",
                size=10,
                color="Black"
            ),
            yaxis_title="RMSE",
            )
    fig.update_xaxes(showticklabels=False)
    fig.write_image(f'benchmark_dim_red/figures/epm/agg_red_dim_{red_dim}.pdf')


In [ ]:
for sample in [100, 200, 500, 1000]:
    for red_dim in [5, 10, 20, 40]:
        for fid in benchs.keys():
            bench = benchs[fid]()
            lb, ub = bench.get_bounds()
            if len(lb) <= red_dim:
                continue
            df = pd.read_parquet(f'benchmark_dim_red/agg_res/epm/{sample}_{fid}.parquet.xz')


            df = df[df['red_dim'] == red_dim]
            df.drop(columns=['fid'], inplace=True)
            
            df['mse'] = np.sqrt(df['mse'])
            df = df.groupby(['method_name', 'cv']).mean().reset_index()

            fig = px.box(df, x='method_name', y='mse', color='method_name',
                            color_discrete_map={
                                'PCA_linear':px.colors.qualitative.Plotly[0],
                                'PCA_rbf':px.colors.qualitative.Dark24[5],
                                'PCA_sigmoid':px.colors.qualitative.Safe[10],
                                'PCA_cosine':px.colors.qualitative.Safe[9],
                                'PCA_poly': px.colors.qualitative.Safe[3],
                                'IPCA':px.colors.qualitative.Safe[5],
                                'FastICA':px.colors.qualitative.Plotly[1],
                                'TruncatedSVD':px.colors.qualitative.Plotly[2],
                                'lolP':px.colors.qualitative.Plotly[3],
                                'isomap':px.colors.qualitative.Plotly[4],
                                'lle':px.colors.qualitative.Plotly[5],
                                'pls_reg':px.colors.qualitative.Plotly[6],
                                'DictionaryLearning':px.colors.qualitative.Plotly[7],
                                'sparsePCA':px.colors.qualitative.Plotly[8],
                                'FA':px.colors.qualitative.Plotly[9],
                                'BernoulliRBM':px.colors.qualitative.G10[4],
                                'GaussianRandomProjection':px.colors.qualitative.G10[8],
                                'SparseRandomProjection':px.colors.qualitative.G10[9],
                            })
            fig.update_xaxes(categoryorder='array', 
                                categoryarray=['PCA_linear', 'PCA_rbf', 'PCA_sigmoid', 
                                            'PCA_cosine', 'PCA_poly', 'IPCA', 'FastICA', 
                                            'DictionaryLearning', 'sparsePCA', 'GaussianRandomProjection', 'SparseRandomProjection', 'lolP', 'TruncatedSVD',
                                            'isomap', 'lle', 'pls_reg', 'FA', 'BernoulliRBM'], title=None)
            fig.update_layout(
                xaxis = dict(
                    tickmode = 'array',
                    tickvals = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24],
                    ticktext =['PCA_linear', 'PCA_rbf', 'PCA_sigmoid', 
                                            'PCA_cosine', 'PCA_poly', 'IPCA', 'FastICA', 
                                            'DictionaryLearning', 'sparsePCA', 'GaussianRandomProjection', 'SparseRandomProjection', 'lolP', 'TruncatedSVD',
                                            'isomap', 'lle', 'pls_reg', 'FA', 'BernoulliRBM']
                )
            )

            fig.update_layout(title_text=None,
                    template='plotly_white',
                    showlegend=False,
                    legend_title=None,
                    width=200, height=120, margin=dict(l=0, r=0, t=0, b=0),
                    font=dict(
                        family="Times New Roman",
                        size=10,
                        color="Black"
                    ),
                    title_font=dict(
                        family="Times New Roman",
                        size=10,
                        color="Black"
                    ),
                    yaxis_title="RMSE",
                    )
            fig.update_xaxes(showticklabels=False)
            fig.write_image(f'benchmark_dim_red/figures/epm/{sample}_{fid}_{red_dim}.pdf')


In [ ]:
for sample in [100, 200, 500, 1000]:  
    dfs = []
    for red_dim in [5, 10, 20, 40]:
        for fid in benchs.keys():
            
            bench = benchs[fid]()
            lb, ub = bench.get_bounds()
            if len(lb) <= red_dim:
                continue
            df = pd.read_parquet(f'benchmark_dim_red/agg_res/epm/{sample}_{fid}.parquet.xz')


            df = df[df['red_dim'] == red_dim]
            df.drop(columns=['fid'], inplace=True)
            
            df['mse'] = np.sqrt(df['mse'])
            df = df.groupby(['method_name', 'cv']).mean().reset_index()
            df['rank'] =1 - ( df['mse'].max() - df['mse'] ) /( df['mse'].max() - df['mse'].min())
            dfs.append(df)

    fig = px.box(df, x='method_name', y='rank', color='method_name',
                    color_discrete_map={
                        'PCA_linear':px.colors.qualitative.Plotly[0],
                        'PCA_rbf':px.colors.qualitative.Dark24[5],
                        'PCA_sigmoid':px.colors.qualitative.Safe[10],
                        'PCA_cosine':px.colors.qualitative.Safe[9],
                        'PCA_poly': px.colors.qualitative.Safe[3],
                        'IPCA':px.colors.qualitative.Safe[5],
                        'FastICA':px.colors.qualitative.Plotly[1],
                        'TruncatedSVD':px.colors.qualitative.Plotly[2],
                        'lolP':px.colors.qualitative.Plotly[3],
                        'isomap':px.colors.qualitative.Plotly[4],
                        'lle':px.colors.qualitative.Plotly[5],
                        'pls_reg':px.colors.qualitative.Plotly[6],
                        'DictionaryLearning':px.colors.qualitative.Plotly[7],
                        'sparsePCA':px.colors.qualitative.Plotly[8],
                        'FA':px.colors.qualitative.Plotly[9],
                        'BernoulliRBM':px.colors.qualitative.G10[4],
                        'GaussianRandomProjection':px.colors.qualitative.G10[8],
                        'SparseRandomProjection':px.colors.qualitative.G10[9],
                    })
    fig.update_xaxes(categoryorder='array', 
                        categoryarray=['PCA_linear', 'PCA_rbf', 'PCA_sigmoid', 
                                    'PCA_cosine', 'PCA_poly', 'IPCA', 'FastICA', 
                                    'DictionaryLearning', 'sparsePCA', 'GaussianRandomProjection', 'SparseRandomProjection', 'lolP', 'TruncatedSVD',
                                    'isomap', 'lle', 'pls_reg', 'FA', 'BernoulliRBM'], title=None)
    fig.update_layout(
        xaxis = dict(
            tickmode = 'array',
            tickvals = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24],
            ticktext =['PCA_linear', 'PCA_rbf', 'PCA_sigmoid', 
                                    'PCA_cosine', 'PCA_poly', 'IPCA', 'FastICA', 
                                    'DictionaryLearning', 'sparsePCA', 'GaussianRandomProjection', 'SparseRandomProjection', 'lolP', 'TruncatedSVD',
                                    'isomap', 'lle', 'pls_reg', 'FA', 'BernoulliRBM']
        )
    )
    fig.add_vline(x=10.5, line_width=1, line_dash="dash", line_color="grey")

    fig.update_layout(title_text=None,
            template='plotly_white',
            showlegend=False,
            legend_title=None,
            width=200, height=200, margin=dict(l=0, r=0, t=0, b=0),
            font=dict(
                family="Times New Roman",
                size=10,
                color="Black"
            ),
            title_font=dict(
                family="Times New Roman",
                size=10,
                color="Black"
            ),
            yaxis_title="Mean Rank",
            )
    fig.update_xaxes(showticklabels=False)
    fig.write_image(f'benchmark_dim_red/figures/epm/real_agg_sample_{sample}.pdf')


In [ ]:
for red_dim in [5, 10, 20, 40]:

    dfs = []
    for sample in [100, 200, 500, 1000]:  

        for fid in benchs.keys():
            
            bench = benchs[fid]()
            lb, ub = bench.get_bounds()
            if len(lb) <= red_dim:
                continue
            df = pd.read_parquet(f'benchmark_dim_red/agg_res/epm/{sample}_{fid}.parquet.xz')


            df = df[df['red_dim'] == red_dim]
            df.drop(columns=['fid'], inplace=True)
            
            df['mse'] = np.sqrt(df['mse'])
            df = df.groupby(['method_name', 'cv']).mean().reset_index()
            df['rank'] =1 - ( df['mse'].max() - df['mse'] ) /( df['mse'].max() - df['mse'].min())
            dfs.append(df)

    fig = px.box(df, x='method_name', y='rank', color='method_name',
                    color_discrete_map={
                        'PCA_linear':px.colors.qualitative.Plotly[0],
                        'PCA_rbf':px.colors.qualitative.Dark24[5],
                        'PCA_sigmoid':px.colors.qualitative.Safe[10],
                        'PCA_cosine':px.colors.qualitative.Safe[9],
                        'PCA_poly': px.colors.qualitative.Safe[3],
                        'IPCA':px.colors.qualitative.Safe[5],
                        'FastICA':px.colors.qualitative.Plotly[1],
                        'TruncatedSVD':px.colors.qualitative.Plotly[2],
                        'lolP':px.colors.qualitative.Plotly[3],
                        'isomap':px.colors.qualitative.Plotly[4],
                        'lle':px.colors.qualitative.Plotly[5],
                        'pls_reg':px.colors.qualitative.Plotly[6],
                        'DictionaryLearning':px.colors.qualitative.Plotly[7],
                        'sparsePCA':px.colors.qualitative.Plotly[8],
                        'FA':px.colors.qualitative.Plotly[9],
                        'BernoulliRBM':px.colors.qualitative.G10[4],
                        'GaussianRandomProjection':px.colors.qualitative.G10[8],
                        'SparseRandomProjection':px.colors.qualitative.G10[9],
                    })
    fig.update_xaxes(categoryorder='array', 
                        categoryarray=['PCA_linear', 'PCA_rbf', 'PCA_sigmoid', 
                                    'PCA_cosine', 'PCA_poly', 'IPCA', 'FastICA', 
                                    'DictionaryLearning', 'sparsePCA', 'GaussianRandomProjection', 'SparseRandomProjection', 'lolP', 'TruncatedSVD',
                                    'isomap', 'lle', 'pls_reg', 'FA', 'BernoulliRBM'], title=None)
    fig.update_layout(
        xaxis = dict(
            tickmode = 'array',
            tickvals = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24],
            ticktext =['PCA_linear', 'PCA_rbf', 'PCA_sigmoid', 
                                    'PCA_cosine', 'PCA_poly', 'IPCA', 'FastICA', 
                                    'DictionaryLearning', 'sparsePCA', 'GaussianRandomProjection', 'SparseRandomProjection', 'lolP', 'TruncatedSVD',
                                    'isomap', 'lle', 'pls_reg', 'FA', 'BernoulliRBM']
        )
    )
    fig.add_vline(x=10.5, line_width=1, line_dash="dash", line_color="grey")

    fig.update_layout(title_text=None,
            template='plotly_white',
            showlegend=False,
            legend_title=None,
            width=200, height=200, margin=dict(l=0, r=0, t=0, b=0),
            font=dict(
                family="Times New Roman",
                size=10,
                color="Black"
            ),
            title_font=dict(
                family="Times New Roman",
                size=10,
                color="Black"
            ),
            yaxis_title="Mean Rank",
            )
    fig.update_xaxes(showticklabels=False)
    fig.write_image(f'benchmark_dim_red/figures/epm/real_red_dim_{red_dim}.pdf')


In [ ]:
for sample in [100, 200, 500, 1000]:
    for fid in range(1, 25):
        for inpt_dim in [20, 40, 60, 120, 240, 500, 1000]:
            data = []
            for instance in range(0, 31):
                f = get_problem(fid=fid, dimension=inpt_dim, instance=instance, problem_class=ProblemClass.BBOB)
                x_orig = create_initial_sample(inpt_dim, sample_type = 'sobol', n=min(50 * inpt_dim, 5000), lower_bound=-5, upper_bound=5, seed=1)

                y = x_orig.apply(lambda u: f(u), axis = 1).values
                for red_dim in [5, 10, 20, 40]:
                    if inpt_dim > red_dim:
                        for method_name, method_class in methods.items():
                            weight = method_name in with_weight
                            kf = KFold(n_splits=5)
                            for cv, (train_index, test_index) in enumerate(kf.split(x_orig)):
                                scaler = MinMaxScaler((1e-5 , 1))
                                y = y.reshape(-1, 1)

                                Ytrain = scaler.fit_transform(y[train_index])
                                Ytrain[Ytrain < 1e-5] = 1e-5 
                                Ytrain = np.log10(Ytrain).reshape(-1)
                                Ytest = scaler.transform(y[test_index])
                                Ytest[Ytest < 1e-5] = 1e-5 
                                Ytest = np.log10(Ytest).reshape(-1)
                                with lzma.open(
                                    f'benchmark_dim_red/{sample}/epm/res/{fid}/{inpt_dim}/{instance}_{method_name}_{red_dim}_{weight}_{cv}.pkl.xz',
                                    "rb",
                                ) as file:
                                    d = pickle.load(file) 
                                    r2 = r2_score(y[test_index], d['preds'])
                                    mse = mean_squared_error(Ytest, d['preds'])
                                    data.append([sample, fid, inpt_dim, instance, method_name, red_dim, weight, cv, r2, mse])
            df = pd.DataFrame(data, columns=['sample', 'fid', 'inpt_dim', 'instance', 'method_name', 'red_dim', 'weight', 'cv', 'r2', 'mse'])
            df.to_parquet(f'/benchmark_dim_red/agg_res/epm/{sample}_{fid}_{inpt_dim}.parquet.xz', compression='lz4')


In [ ]:
d['mse']

In [ ]:
df = pd.read_parquet(f'/benchmark_dim_red/agg_res/epm/100_3_40.parquet.xz')

In [ ]:
df[df['red_dim'] == 20]

In [ ]:
sample = 100
fid = 1
inpt_dim = 20
df = pd.read_parquet(f'benchmark_dim_red/agg_res/epm/{sample}_{fid}_{inpt_dim}.parquet.xz')

df = df[df['red_dim'] == red_dim]
df['mse'] = np.sqrt(df['mse'])
df = df.groupby(['method_name', 'cv']).mean().reset_index()

fig = px.box(df, x='method_name', y='mse', color='method_name',
                color_discrete_map={
                    'PCA_linear':px.colors.qualitative.Plotly[0],
                    'PCA_rbf':px.colors.qualitative.Dark24[5],
                    'PCA_sigmoid':px.colors.qualitative.Safe[10],
                    'PCA_cosine':px.colors.qualitative.Safe[9],
                    'PCA_poly': px.colors.qualitative.Safe[3],
                    'IPCA':px.colors.qualitative.Safe[5],
                    'FastICA':px.colors.qualitative.Plotly[1],
                    'TruncatedSVD':px.colors.qualitative.Plotly[2],
                    'lolP':px.colors.qualitative.Plotly[3],
                    'isomap':px.colors.qualitative.Plotly[4],
                    'lle':px.colors.qualitative.Plotly[5],
                    'pls_reg':px.colors.qualitative.Plotly[6],
                    'DictionaryLearning':px.colors.qualitative.Plotly[7],
                    'sparsePCA':px.colors.qualitative.Plotly[8],
                    'FA':px.colors.qualitative.Plotly[9],
                    'BernoulliRBM':px.colors.qualitative.G10[4],
                    'GaussianRandomProjection':px.colors.qualitative.G10[8],
                    'SparseRandomProjection':px.colors.qualitative.G10[9],
                })
fig.update_xaxes(categoryorder='array', 
                    categoryarray=['PCA_linear', 'PCA_rbf', 'PCA_sigmoid', 
                                'PCA_cosine', 'PCA_poly', 'IPCA', 'FastICA', 
                                'DictionaryLearning', 'sparsePCA', 'GaussianRandomProjection', 'SparseRandomProjection', 'lolP', 'TruncatedSVD',
                                'isomap', 'lle', 'pls_reg', 'FA', 'BernoulliRBM'], title=None)
fig.update_layout(
    xaxis = dict(
        tickmode = 'array',
        tickvals = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24],
        ticktext =['PCA_linear', 'PCA_rbf', 'PCA_sigmoid', 
                                'PCA_cosine', 'PCA_poly', 'IPCA', 'FastICA', 
                                'DictionaryLearning', 'sparsePCA', 'GaussianRandomProjection', 'SparseRandomProjection', 'lolP', 'TruncatedSVD',
                                'isomap', 'lle', 'pls_reg', 'FA', 'BernoulliRBM']
    )
)
fig.add_vline(x=10.5, line_width=1, line_dash="dash", line_color="grey")

fig.update_layout(title_text=None,
        template='plotly_white',
        showlegend=False,
        legend_title=None,
        width=400, height=300, margin=dict(l=0, r=0, t=0, b=0),
        font=dict(
            family="Times New Roman",
            size=10,
            color="Black"
        ),
        title_font=dict(
            family="Times New Roman",
            size=10,
            color="Black"
        ),
        yaxis_title="RMSE",
        )